# Home Credit Risk Analysis

## Dataset: installment_payments.csv

### Objetivo

Analizar la estructura, calidad y relaciones del dataset para comprender su papel dentro de la arquitectura Lakehouse y del modelo dimensional.

---

### Contexto del negocio

El dataset installments_payments contiene el historial de pagos realizados por los clientes para las diferentes cuotas asociadas a créditos previamente otorgados.

Cada registro representa un evento de pago relacionado con una cuota específica y permite analizar el comportamiento real de cumplimiento de las obligaciones financieras.

La información almacenada resulta fundamental para evaluar patrones de pago, atrasos, cumplimiento contractual y riesgo crediticio.

---

### Rol dentro de la arquitectura

application_train
        │
        ▼
previous_application
        │
        ▼
installments_payments

# 1. Importación de librerías

Se cargan las librerías necesarias para el análisis exploratorio de datos.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# 2. Carga del Dataset

Se carga el dataset para realizar el análisis exploratorio y comprender su estructura, calidad y relevancia dentro del dominio de negocio.

In [2]:
df = pd.read_csv("../../../data/raw/homeCredit/installments_payments.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/raw/homeCredit/installments_payments.csv'

# 3. Tamaño del Dataset

## Resultados

| Métrica | Valor |
|----------|---------|
| Filas | 13,605,401 |
| Columnas | 8 |

## Interpretación

El dataset `installments_payments.csv` contiene 13,605,401 registros distribuidos en únicamente 8 columnas.

A pesar de poseer una estructura relativamente compacta, el elevado volumen de registros evidencia que se trata de una fuente de datos transaccional orientada al seguimiento detallado de eventos de pago.

Cada registro representa una interacción financiera asociada al cumplimiento de una cuota específica dentro de un crédito previamente otorgado.

---

### Características Generales

El dataset almacena información relacionada con:

- Cuotas programadas.
- Fechas de vencimiento.
- Fechas reales de pago.
- Montos esperados.
- Montos efectivamente pagados.
- Historial de cumplimiento financiero.

---

### Relevancia para el Negocio

La información contenida en este dataset permite analizar el comportamiento real de pago de los clientes.

A diferencia de otros datasets que describen solicitudes o créditos otorgados, installments_payments registra eventos financieros reales ocurridos durante la vida del crédito.

Esto permite evaluar:

- Cumplimiento de pagos.
- Atrasos.
- Adelantos.
- Pagos parciales.
- Comportamiento financiero histórico.

---

### Relevancia para el Riesgo Crediticio

Este dataset constituye una de las principales fuentes para la construcción de indicadores de riesgo, ya que refleja el comportamiento observado del cliente frente a sus obligaciones financieras.

A partir de esta información será posible construir métricas relacionadas con:

- Días promedio de atraso.
- Frecuencia de incumplimiento.
- Porcentaje de pagos puntuales.
- Número de cuotas pagadas.
- Severidad de mora.
- Historial de cumplimiento.

---

### Consideraciones para Ingeniería de Datos

Con más de 13 millones de registros, installments_payments se posiciona como uno de los datasets más voluminosos del proyecto.

Esta característica lo convierte en un candidato natural para procesamiento distribuido mediante Spark y para la generación de agregaciones históricas dentro de las capas Silver, Intermediate y Gold de la arquitectura Lakehouse.

# 4. Vista Preliminar

Se inspeccionan los primeros registros del dataset con el objetivo de comprender su estructura, contenido y nivel de granularidad.

## Observaciones Iniciales

La muestra evidencia que cada registro representa un evento de pago asociado a una cuota específica de un crédito previamente otorgado.

El dataset almacena información tanto de las condiciones programadas de pago como de los pagos efectivamente realizados por los clientes.

---

## Variables Observadas

| Variable | Descripción |
|-----------|-------------|
| SK_ID_PREV | Identificador del crédito asociado |
| SK_ID_CURR | Identificador del cliente |
| NUM_INSTALMENT_VERSION | Versión del plan de cuotas |
| NUM_INSTALMENT_NUMBER | Número de cuota |
| DAYS_INSTALMENT | Fecha programada de pago |
| DAYS_ENTRY_PAYMENT | Fecha real de pago |
| AMT_INSTALMENT | Valor esperado de la cuota |
| AMT_PAYMENT | Valor efectivamente pagado |

---

## Hallazgos Iniciales

La muestra permite identificar diferencias entre:

### Fechas Programadas y Fechas Reales

Las variables:

- DAYS_INSTALMENT
- DAYS_ENTRY_PAYMENT

permiten comparar la fecha esperada de pago con la fecha en que realmente se realizó la transacción.

Por ejemplo:

| Cuota | Fecha Programada | Fecha Real |
|---------|---------|---------|
| 6 | -1180 | -1187 |

En este caso el pago fue realizado antes de la fecha esperada.

---

### Montos Esperados y Montos Pagados

Las variables:

- AMT_INSTALMENT
- AMT_PAYMENT

permiten evaluar el nivel de cumplimiento financiero.

Por ejemplo:

| Valor Esperado | Valor Pagado |
|---------------|-------------|
| 2165.040 | 2160.585 |

Lo anterior evidencia que pueden existir diferencias entre el valor programado y el valor efectivamente abonado.

---

## Potenciales Indicadores

La estructura observada permitirá construir métricas relacionadas con:

- Días de atraso.
- Días de anticipación.
- Cumplimiento de pagos.
- Pagos parciales.
- Pagos completos.
- Historial de mora.
- Frecuencia de incumplimiento.

---

## Primera Hipótesis de Granularidad

A partir de los registros observados se plantea la siguiente hipótesis:

1 registro = 1 cuota pagada

Esta hipótesis será validada posteriormente mediante el análisis de claves y cardinalidad.

---

## Relación con previous_application

Mientras que `previous_application.csv` almacena la información asociada a las solicitudes y créditos otorgados, `installments_payments.csv` registra el comportamiento real de pago observado durante la vida del crédito.

Esta característica convierte a installments_payments en una de las principales fuentes para la construcción de indicadores históricos de riesgo crediticio.

In [3]:
df.head()

NameError: name 'df' is not defined

# 5. Estructura General

Se analiza la composición estructural del dataset para comprender los tipos de datos utilizados, evaluar la calidad general del esquema y estimar la complejidad de los procesos analíticos posteriores.

---

## Resumen Estructural

| Métrica | Valor |
|----------|---------|
| Registros | 13,605,401 |
| Columnas | 8 |
| Variables Numéricas | 8 |
| Variables Categóricas | 0 |
| Tipos de Datos Distintos | 2 |
| Consumo de Memoria | 830.4 MB |

---

## Distribución de Tipos de Datos

| Tipo de Dato | Cantidad |
|--------------|----------|
| float64 | 5 |
| int64 | 3 |

---

## Interpretación

El dataset presenta una estructura completamente numérica compuesta por variables financieras, temporales e identificadores de negocio.

A diferencia de otros datasets analizados previamente, no se identifican variables categóricas, lo que sugiere que esta fuente está orientada exclusivamente al registro de eventos transaccionales relacionados con pagos de cuotas.

---

## Componentes Principales

### Variables Identificadoras

Las columnas:

- SK_ID_PREV
- SK_ID_CURR

permiten relacionar cada evento de pago con el crédito correspondiente y con el cliente asociado.

---

### Variables de Cuotas

Las columnas:

- NUM_INSTALMENT_VERSION
- NUM_INSTALMENT_NUMBER

permiten identificar la estructura de pagos asociada a cada crédito.

Estas variables facilitan el seguimiento de modificaciones en los planes de pago y el análisis secuencial de cuotas.

---

### Variables Temporales

Las columnas:

- DAYS_INSTALMENT
- DAYS_ENTRY_PAYMENT

permiten comparar la fecha programada de pago con la fecha real de cumplimiento.

Estas variables constituyen la base para la construcción de indicadores relacionados con mora, puntualidad y comportamiento financiero.

---

### Variables Financieras

Las columnas:

- AMT_INSTALMENT
- AMT_PAYMENT

representan los montos esperados y efectivamente pagados por los clientes.

Su análisis permitirá identificar pagos completos, pagos parciales, incumplimientos y posibles sobrepagos.

---

## Calidad Estructural

La estructura observada es consistente con una tabla transaccional orientada al registro detallado de pagos.

La ausencia de variables categóricas simplifica los procesos de transformación y agregación, favoreciendo la construcción de métricas históricas dentro de las capas Silver, Intermediate y Gold.

---

## Consideraciones para Ingeniería de Datos

Con más de 13 millones de registros y una estructura completamente numérica, installments_payments constituye una fuente ideal para procesamiento distribuido mediante Spark.

Su diseño facilita la generación de indicadores agregados asociados al comportamiento de pago, cumplimiento financiero y riesgo crediticio.

In [4]:
df.info()

NameError: name 'df' is not defined

In [5]:
df.dtypes.value_counts()

NameError: name 'df' is not defined

# 6. Identificación de Claves

Se identifican:

- Clave primaria
- Claves foráneas
- Relaciones con otros datasets

In [6]:
len(df)

NameError: name 'df' is not defined

In [7]:
df["SK_ID_PREV"].nunique()

NameError: name 'df' is not defined

In [8]:
df["SK_ID_CURR"].nunique()

NameError: name 'df' is not defined

In [9]:
df.duplicated().sum()

NameError: name 'df' is not defined

In [10]:
df[["SK_ID_PREV", "NUM_INSTALMENT_NUMBER", "NUM_INSTALMENT_VERSION"]].duplicated().sum()

NameError: name 'df' is not defined

### Clave Primaria

No se identificó una clave primaria simple ni compuesta que garantice unicidad absoluta a nivel de registro.

Las pruebas realizadas sobre diferentes combinaciones de columnas mostraron que múltiples eventos de pago pueden estar asociados a una misma cuota dentro de un crédito.

Esto indica que el dataset almacena eventos transaccionales de pago y no únicamente cuotas programadas.

---

### Interpretación

La granularidad observada sugiere que un mismo crédito puede presentar:

- Varias versiones del plan de cuotas.
- Múltiples cuotas.
- Múltiples eventos de pago asociados a una misma cuota.

Por esta razón, la unicidad del registro depende de una combinación más amplia de atributos operativos que no se encuentra explícitamente representada mediante una clave única.

---

### Clave Foránea

SK_ID_CURR

Permite relacionar cada evento de pago con el cliente correspondiente.

Asimismo:

SK_ID_PREV

permite relacionar cada pago con el crédito asociado.

### Cardinalidad

application_train (1)

↓

previous_application (N)

↓

installments_payments (N)

Un cliente puede tener múltiples créditos.

Cada crédito puede tener múltiples cuotas.

Cada cuota puede registrar múltiples eventos de pago.

# 7. Calidad de Datos

Se analiza la completitud de la información y posibles problemas de calidad.

In [11]:
null_percentage = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

null_percentage.head(20)

NameError: name 'df' is not defined

In [12]:
high_nulls = null_percentage[null_percentage > 50]

medium_nulls = null_percentage[
    (null_percentage > 20)
    & (null_percentage <= 50)
]

low_nulls = null_percentage[
    (null_percentage > 0)
    & (null_percentage <= 20)
]

print("Columnas > 50%:", len(high_nulls))
print("Columnas entre 20%-50%:", len(medium_nulls))
print("Columnas < 20%:", len(low_nulls))

NameError: name 'null_percentage' is not defined

## Interpretación

El análisis de calidad de datos evidencia un nivel de completitud excepcional dentro del dataset.

De las ocho variables analizadas, seis presentan un 100% de completitud y únicamente dos columnas registran valores faltantes.

---

### Valores Nulos Identificados

| Columna | % Nulos |
|----------|---------|
| DAYS_ENTRY_PAYMENT | 0.021% |
| AMT_PAYMENT | 0.021% |

---

### Distribución de Completitud

| Categoría | Cantidad |
|------------|----------|
| Columnas > 50% nulos | 0 |
| Columnas entre 20% y 50% | 0 |
| Columnas < 20% | 2 |

---

### Interpretación de Negocio

Los valores faltantes se concentran exclusivamente en:

- DAYS_ENTRY_PAYMENT
- AMT_PAYMENT

Ambas variables representan información asociada al pago efectivamente realizado por el cliente.

La coincidencia exacta en el porcentaje de valores faltantes sugiere una relación directa entre ambas columnas.

Esto indica que los registros sin fecha de pago también carecen de monto pagado registrado.

---

### Hipótesis de Negocio

Los valores faltantes probablemente corresponden a cuotas que:

- No han sido pagadas.
- Se encontraban pendientes al momento de la extracción.
- Presentaban eventos de pago aún no registrados.

Por lo tanto, estos nulos podrían representar información de negocio relevante y no necesariamente errores de calidad de datos.

---

### Hallazgos

- Más del 99.97% de los registros presentan información completa.
- No se identifican problemas significativos de completitud.
- Los valores faltantes se concentran únicamente en variables relacionadas con pagos realizados.
- La calidad observada facilita la construcción de indicadores financieros y de riesgo.

---

### Implicaciones para la Arquitectura Lakehouse

#### Bronze

Los registros deben conservarse sin modificaciones para preservar la trazabilidad de los eventos de pago.

#### Silver

Será necesario validar si los registros con valores faltantes corresponden efectivamente a cuotas no pagadas o a situaciones operativas específicas.

#### Gold

Los eventos sin pago registrado pueden convertirse en señales relevantes para la construcción de métricas asociadas al incumplimiento financiero y al comportamiento de mora.

---

### Conclusión

El dataset installments_payments presenta uno de los niveles de calidad más altos observados hasta el momento dentro del proyecto.

La baja proporción de valores faltantes y la consistencia estructural de sus variables lo convierten en una fuente altamente confiable para la construcción de indicadores históricos de cumplimiento y riesgo crediticio.

# 8. Detección de Registros Duplicados

In [13]:
df.duplicated().sum()

NameError: name 'df' is not defined

## Interpretación

Se realizó una validación de duplicidad sobre la totalidad de los registros presentes en el dataset.

---

### Resultados

| Métrica | Valor |
|----------|---------|
| Registros Totales | 13,605,401 |
| Registros Duplicados | 0 |
| Porcentaje de Duplicados | 0% |

---

### Hallazgos

No se identificaron registros completamente duplicados dentro del dataset.

Este resultado confirma la consistencia estructural de la información almacenada y garantiza que cada evento de pago representa una observación única dentro del historial transaccional.

---

### Interpretación de Negocio

La ausencia de duplicados resulta especialmente relevante debido a la naturaleza transaccional del dataset.

Cada registro representa un evento de pago asociado a una cuota específica y su correcta identificación es fundamental para el cálculo de métricas financieras y de riesgo.

La inexistencia de duplicados evita sesgos en indicadores relacionados con:

- Cumplimiento de pagos.
- Frecuencia de mora.
- Días de atraso.
- Montos pagados.
- Historial financiero del cliente.

---

### Implicaciones para la Arquitectura Lakehouse

#### Bronze

Los registros pueden ser almacenados sin necesidad de aplicar procesos de deduplicación.

#### Silver

Los esfuerzos de validación podrán concentrarse en reglas de calidad, consistencia temporal y comportamiento de pago.

#### Gold

La ausencia de duplicados garantiza la confiabilidad de los indicadores históricos construidos a partir de eventos de pago.

---

### Conclusión

El dataset presenta una estructura consistente y libre de registros duplicados, fortaleciendo su confiabilidad como fuente principal para el análisis de cumplimiento financiero y comportamiento crediticio.

# 9. Variables de Negocio Relevantes

Se analizan las variables con mayor relevancia para el dominio financiero.

In [14]:
important_cols = [
    "SK_ID_PREV",
    "SK_ID_CURR",
    "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT",
    "AMT_PAYMENT"
]

df[important_cols].head()

NameError: name 'df' is not defined

## Interpretación

Las variables seleccionadas representan los elementos fundamentales para analizar el comportamiento de pago de los clientes y evaluar el cumplimiento de sus obligaciones financieras.

A diferencia de otros datasets del proyecto, installments_payments registra eventos reales de pago, permitiendo observar directamente cómo los clientes gestionan sus compromisos crediticios.

---

### Variables Identificadas

| Variable | Descripción |
|-----------|-------------|
| SK_ID_PREV | Identificador del crédito asociado |
| SK_ID_CURR | Identificador del cliente |
| NUM_INSTALMENT_NUMBER | Número de cuota |
| DAYS_INSTALMENT | Fecha programada de pago |
| DAYS_ENTRY_PAYMENT | Fecha real de pago |
| AMT_INSTALMENT | Valor esperado de la cuota |
| AMT_PAYMENT | Valor efectivamente pagado |

---

### Relevancia de Negocio

#### SK_ID_PREV

Permite asociar cada evento de pago con el crédito correspondiente.

Esta variable facilita la reconstrucción del historial financiero de cada operación crediticia.

---

#### SK_ID_CURR

Permite relacionar los eventos de pago con el cliente correspondiente.

Será utilizada posteriormente para construir métricas agregadas de comportamiento financiero a nivel de cliente.

---

#### NUM_INSTALMENT_NUMBER

Representa la secuencia de cuotas dentro del crédito.

Permite analizar la evolución del comportamiento de pago a lo largo del tiempo.

---

#### DAYS_INSTALMENT

Representa la fecha programada para el pago de la cuota.

Constituye la referencia utilizada para determinar cumplimiento o atraso.

---

#### DAYS_ENTRY_PAYMENT

Representa la fecha real en la que se registró el pago.

Su comparación con DAYS_INSTALMENT permitirá medir puntualidad y mora.

---

#### AMT_INSTALMENT

Corresponde al valor esperado de la cuota.

Representa la obligación financiera originalmente pactada.

---

#### AMT_PAYMENT

Corresponde al valor efectivamente pagado por el cliente.

Permite evaluar cumplimiento financiero y detectar pagos parciales o diferencias respecto al valor esperado.

---

### Hallazgos Iniciales

La muestra evidencia que existen escenarios donde:

- El pago se realiza antes de la fecha programada.
- El pago se realiza exactamente en la fecha esperada.
- El pago se realiza después de la fecha programada.
- El valor pagado puede diferir ligeramente del valor esperado.

Estas situaciones representan señales relevantes para la evaluación del comportamiento crediticio.

---

### Indicadores Potenciales

A partir de estas variables será posible construir métricas como:

- Días de atraso.
- Días de anticipación.
- Porcentaje de pagos puntuales.
- Frecuencia de mora.
- Pago promedio por cuota.
- Cumplimiento financiero.
- Historial de comportamiento de pago.

---

### Conclusión

Las variables analizadas constituyen la base para la construcción de indicadores históricos de cumplimiento financiero.

Su combinación permitirá medir de forma precisa la capacidad y disciplina de pago de los clientes, convirtiendo a installments_payments en una de las principales fuentes de información para el análisis de riesgo crediticio dentro del proyecto.

# 10. Variables Categóricas

In [15]:
categorical_columns = df.select_dtypes(include=["object", "string"])

categorical_columns.columns.tolist()

NameError: name 'df' is not defined

In [16]:
for col in categorical_columns.columns:
    print(f"{col}: {df[col].nunique()}")

NameError: name 'categorical_columns' is not defined

## Interpretación

Se realizó la identificación de variables categóricas dentro del dataset mediante el análisis de los tipos de datos presentes.

### Resultados

No se identificaron variables categóricas dentro del dataset.

| Métrica | Valor |
|----------|---------|
| Variables categóricas | 0 |

---

### Interpretación

La totalidad de las columnas presentes en installments_payments corresponden a variables numéricas.

Esta característica diferencia a este dataset de otras fuentes analizadas previamente, las cuales incorporaban atributos descriptivos asociados a productos financieros, estados de solicitud o características de clientes.

---

### Implicaciones Analíticas

La ausencia de variables categóricas indica que el dataset está orientado exclusivamente al registro de eventos transaccionales relacionados con pagos.

La información disponible se centra en:

- Identificadores de negocio.
- Fechas relativas.
- Valores financieros.
- Estructura de cuotas.

---

### Ventajas para Ingeniería de Datos

La estructura completamente numérica simplifica:

- Procesos de validación.
- Agregaciones.
- Cálculos estadísticos.
- Transformaciones en Spark.
- Construcción de métricas históricas.

Asimismo, reduce la necesidad de procesos de codificación o normalización de atributos categóricos.

---

### Conclusión

La ausencia de variables categóricas confirma que installments_payments constituye una tabla transaccional especializada en el seguimiento de pagos realizados por los clientes.

Su estructura está diseñada para medir comportamiento financiero observado y servirá como una de las principales fuentes para la construcción de indicadores de cumplimiento y riesgo crediticio.

# 11. Relaciones del Dataset

## Interpretación

El dataset `installments_payments.csv` registra los eventos de pago asociados a los créditos previamente otorgados a los clientes.

Su función principal dentro del ecosistema Home Credit es proporcionar evidencia del comportamiento financiero observado durante la vida de los créditos.

---

### Relación Principal

El dataset se relaciona con las demás fuentes mediante dos identificadores clave:

| Variable | Relación |
|-----------|----------|
| SK_ID_CURR | Cliente |
| SK_ID_PREV | Crédito previamente otorgado |

---

### Relación con application_train

application_train (1)

↓

installments_payments (N)

Un cliente puede registrar múltiples eventos de pago a lo largo de su historial financiero.

---

### Relación con previous_application

previous_application (1)

↓

installments_payments (N)

Un crédito previamente aprobado puede generar múltiples cuotas y múltiples eventos de pago durante su ciclo de vida.

---

### Jerarquía de Información

Cliente

↓

Solicitud Actual

↓

Crédito Histórico

↓

Cuotas

↓

Pagos

Esta estructura representa el flujo natural del ciclo crediticio dentro de Home Credit.

---

### Granularidad

1 registro = 1 evento de pago asociado a una cuota

La granularidad observada confirma que installments_payments constituye una tabla transaccional orientada al seguimiento detallado del comportamiento financiero de los clientes.

---

### Integración con el Modelo Dimensional

El dataset constituye una de las principales fuentes para la construcción de métricas históricas relacionadas con:

- Cumplimiento financiero.
- Historial de pagos.
- Mora.
- Atrasos.
- Severidad de incumplimiento.
- Frecuencia de pago.

---

### Relevancia para la Arquitectura Lakehouse

Durante las etapas Silver e Intermediate, este dataset permitirá consolidar indicadores históricos de comportamiento financiero.

Posteriormente, dichas métricas enriquecerán las capas Gold y Diamond mediante atributos orientados a la evaluación del riesgo crediticio.

---

### Conclusión

installments_payments representa la fuente más cercana al comportamiento financiero real observado dentro del ecosistema Home Credit.

Su integración con application_train y previous_application permitirá construir una visión completa del historial de cumplimiento de los clientes.N]

# 12. Conclusiones Técnicas

## Resumen Ejecutivo

El dataset `installments_payments.csv` almacena el historial de pagos realizados por los clientes para las cuotas asociadas a créditos previamente otorgados.

Su principal objetivo es registrar el comportamiento financiero observado durante la ejecución de los contratos crediticios.

A diferencia de otros datasets analizados previamente, esta fuente refleja eventos reales de pago y constituye una de las principales bases para la construcción de indicadores de riesgo crediticio.

---

## Métricas Generales

| Métrica | Valor |
|----------|---------|
| Registros | 13,605,401 |
| Columnas | 8 |
| Variables Numéricas | 8 |
| Variables Categóricas | 0 |
| Duplicados | 0 |
| Consumo de Memoria | 830.4 MB |

---

## Calidad de Datos

El dataset presenta un nivel de calidad excepcional.

Las validaciones realizadas evidenciaron:

- Ausencia de registros duplicados.
- Ausencia de problemas significativos de completitud.
- Consistencia estructural entre variables relacionadas.

Los únicos valores faltantes identificados corresponden a:

- DAYS_ENTRY_PAYMENT
- AMT_PAYMENT

Ambas columnas presentan únicamente un 0.021% de valores nulos.

---

## Granularidad

Cada registro representa:

**1 evento de pago asociado a una cuota específica**

Esta granularidad convierte a installments_payments en una tabla transaccional orientada al seguimiento detallado de pagos.

---

## Relaciones Identificadas

### Relación con application_train

application_train (1)

↓

installments_payments (N)

---

### Relación con previous_application

previous_application (1)

↓

installments_payments (N)

Estas relaciones permiten reconstruir el historial completo de comportamiento financiero de cada cliente.

---

## Variables de Negocio Relevantes

Las variables de mayor valor analítico identificadas fueron:

- DAYS_INSTALMENT
- DAYS_ENTRY_PAYMENT
- AMT_INSTALMENT
- AMT_PAYMENT
- NUM_INSTALMENT_NUMBER

Estas variables permiten evaluar el cumplimiento de las obligaciones financieras y construir indicadores asociados al riesgo crediticio.

---

## Hallazgos Principales

### Calidad de los Datos

Más del 99.97% de los registros presentan información completa.

No se identificaron problemas significativos de integridad ni duplicidad.

---

### Estructura Transaccional

El dataset no contiene variables categóricas.

La totalidad de la información se encuentra representada mediante atributos numéricos asociados a pagos, cuotas y fechas.

Esta característica facilita la construcción de métricas agregadas y procesos analíticos de alto rendimiento.

---

### Valor Analítico

La combinación de fechas programadas y fechas reales de pago permite analizar:

- Puntualidad.
- Atrasos.
- Anticipaciones.
- Cumplimiento financiero.

Asimismo, la comparación entre montos esperados y montos pagados permite evaluar:

- Pagos completos.
- Pagos parciales.
- Diferencias financieras.

---

## Rol dentro del Modelo Dimensional

El dataset es candidato natural para la construcción de una tabla de hechos asociada al comportamiento de pago.

### FACT_INSTALLMENTS_PAYMENTS

A partir de esta información podrán construirse indicadores como:

- Total de cuotas pagadas.
- Total pagado por cliente.
- Días promedio de atraso.
- Frecuencia de mora.
- Porcentaje de pagos puntuales.
- Historial de cumplimiento financiero.

---

## Rol dentro de la Arquitectura Lakehouse

installments_payments.csv

↓

Bronze (ingesta cruda)

↓

Silver (validación y normalización)

↓

Intermediate (agregaciones históricas)

↓

Gold (indicadores de comportamiento financiero)

↓

Diamond (convergencia corporativa)

---

## Recomendaciones para la Siguiente Fase

1. Integrar installments_payments con previous_application mediante SK_ID_PREV.
2. Consolidar métricas históricas por cliente utilizando SK_ID_CURR.
3. Diseñar indicadores asociados a mora y cumplimiento financiero.
4. Incorporar métricas derivadas en las capas Gold y Diamond.
5. Implementar reglas de calidad asociadas a pagos pendientes y eventos sin registro de pago.

---

## Conclusión Final

installments_payments constituye una de las fuentes más valiosas del ecosistema Home Credit para la evaluación del comportamiento financiero observado.

Su volumen, calidad y naturaleza transaccional permiten construir indicadores robustos de cumplimiento, mora y riesgo crediticio, convirtiéndolo en un componente fundamental para las futuras capas analíticas de la arquitectura Lakehouse.